# 02 Results and Final Fit

연구 목적별 통합 Notebook — 승인된 매핑, 원본 스크립트 및 결과는 변경하지 않는다. 1차 Fine-Tuning까지 포함. 2차 Fine-Tuning 제외.

## 연구 구성
- [`fit_frozen.py`](https://github.com/ryu980920/Machine_Learning/blob/ec30f7fd4a97de9f8336df2eef6a52b394c810c0/05_model_optimize/Extra_Trees/Normal/scripts/fit_frozen.py) — [실험 기록/결과 위치](https://github.com/ryu980920/Machine_Learning/tree/ec30f7fd4a97de9f8336df2eef6a52b394c810c0/05_model_optimize/Extra_Trees/Normal/logs)
- [`aggregate.py`](https://github.com/ryu980920/Machine_Learning/blob/ec30f7fd4a97de9f8336df2eef6a52b394c810c0/05_model_optimize/Extra_Trees/Normal/scripts/aggregate.py) — [실험 기록/결과 위치](https://github.com/ryu980920/Machine_Learning/tree/ec30f7fd4a97de9f8336df2eef6a52b394c810c0/05_model_optimize/Extra_Trees/Normal/logs)
- [`render_report.py`](https://github.com/ryu980920/Machine_Learning/blob/ec30f7fd4a97de9f8336df2eef6a52b394c810c0/05_model_optimize/Extra_Trees/Normal/scripts/render_report.py) — [실험 기록/결과 위치](https://github.com/ryu980920/Machine_Learning/tree/ec30f7fd4a97de9f8336df2eef6a52b394c810c0/05_model_optimize/Extra_Trees/Normal/logs)
- [`write_log.py`](https://github.com/ryu980920/Machine_Learning/blob/ec30f7fd4a97de9f8336df2eef6a52b394c810c0/05_model_optimize/Extra_Trees/Normal/scripts/write_log.py) — [실험 기록/결과 위치](https://github.com/ryu980920/Machine_Learning/tree/ec30f7fd4a97de9f8336df2eef6a52b394c810c0/05_model_optimize/Extra_Trees/Normal/logs)

## 사용법
원본 코드 셀은 `%%archive_source`로 등록하는 실제 원본 코드이며, 함수명 충돌을 막기 위해 현재 커널에서 임의 실행하지 않는다. 맨 아래의 실행 셀에서 원본 스크립트/Notebook을 고정 커밋의 **별도 checkout**에서 실행한다. 각 실험의 실행 인자·입력·사전 조건은 위 원본 로그/코드로 확인한다. 기존 로그·결과와 새 출력이 섞이지 않도록 독립 복사본을 사용한다. Original Test/교수님 Test를 포함한 파일은 실행 환경에 두지 않는다. 과거 스크립트가 해당 자료를 요구할 경우 실행하지 않는다. 자동 재실행 및 성능 동등성은 미검증.

원본 실행을 위한 로그 디렉터리를 별도로 지정한다.

In [ ]:
# 공통 실행 제어 — Run All은 원본 코드 등록만 하며 학습하지 않는다.
from pathlib import Path
import hashlib, json, subprocess, sys, datetime
from IPython import get_ipython
SOURCE_CELLS = {}
def archive_source(line, cell):
    SOURCE_CELLS[line.strip()] = cell
get_ipython().register_magic_function(archive_source, magic_kind="cell", magic_name="archive_source")
# 실행 시에만 설정: 기존 작업 트리가 아닌 임시 checkout (git worktree add --detach ... <ref>)
WORKTREE = None
LOG_DIR = None
RUN_PLAN = []  # 예: [("원본/스크립트.py", ["--인자", "값"])]
ALLOW_EXECUTION = False

def blob_sha(data):
    return hashlib.sha1(f"blob {len(data)}\0".encode()+data).hexdigest()

def get_verified_source(path):
    if WORKTREE is None:
        raise ValueError("해당 커밋의 별도 checkout 경로 WORKTREE를 지정하세요.")
    root=Path(WORKTREE).expanduser().resolve()
    entry=SOURCE_INDEX[path]
    head=subprocess.check_output(["git","-C",str(root),"rev-parse","HEAD"],text=True).strip()
    if head != entry["ref"]:
        raise ValueError(f"checkout ref 불일치: {head} != {entry['ref']}")
    p=root/path
    if blob_sha(p.read_bytes()) != entry["sha"]:
        raise ValueError(f"원본 Git blob 불일치: {path}")
    if path in SOURCE_CELLS:
        if p.read_text(encoding="utf-8-sig").rstrip("\n") != SOURCE_CELLS[path].rstrip("\n").lstrip("\ufeff"):
            raise ValueError(f"Notebook 보존 코드와 원본이 다릅니다: {path}")
    return root,p

def show_original(path):
    from IPython.display import display, Code
    _,p=get_verified_source(path)
    display(Code(p.read_text(encoding="utf-8"),language="python" if p.suffix==".py" else "json"))

def execute_original(path, args):
    if not ALLOW_EXECUTION:
        raise RuntimeError("명시적 실행 승인이 필요합니다.")
    root,p=get_verified_source(path)
    if LOG_DIR is None:
        raise ValueError("새 외부 LOG_DIR를 지정하세요.")
    logs=Path(LOG_DIR).expanduser().resolve();logs.mkdir(parents=True,exist_ok=True)
    if logs==root or root in logs.parents:
        raise ValueError("LOG_DIR는 checkout 밖에 두세요.")
    stamp=datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
    if p.suffix==".ipynb":
        cmd=[sys.executable,"-m","jupyter","nbconvert","--to","notebook","--execute",str(p),"--output-dir",str(logs),"--output",f"{p.stem}_{stamp}.ipynb",*args]
    else:
        cmd=[sys.executable,str(p),*args]
    log=logs/f"{p.stem}_{stamp}.log"
    with log.open("x",encoding="utf-8") as f:
        f.write(json.dumps({"source":path,**SOURCE_INDEX[path],"command":cmd,"cwd":str(p.parent)},ensure_ascii=False)+"\n");f.flush()
        result=subprocess.run(cmd,cwd=p.parent,stdout=f,stderr=subprocess.STDOUT)
        f.write(f"\nreturncode={result.returncode}\n")
    print(log)
    result.check_returncode()


In [ ]:
SOURCE_INDEX = {'05_model_optimize/Extra_Trees/Normal/scripts/fit_frozen.py': {'ref': 'ec30f7fd4a97de9f8336df2eef6a52b394c810c0', 'sha': '60627ee546d0ac9be4b9e01d7e70f65a698edd7d'}, '05_model_optimize/Extra_Trees/Normal/scripts/aggregate.py': {'ref': 'ec30f7fd4a97de9f8336df2eef6a52b394c810c0', 'sha': '14dab185b29c7e051ed8b542068781684ce4307b'}, '05_model_optimize/Extra_Trees/Normal/scripts/render_report.py': {'ref': 'ec30f7fd4a97de9f8336df2eef6a52b394c810c0', 'sha': 'b92c5cc42c0134974d78907ecd2c3dc4e3794201'}, '05_model_optimize/Extra_Trees/Normal/scripts/write_log.py': {'ref': 'ec30f7fd4a97de9f8336df2eef6a52b394c810c0', 'sha': '65deaa3ddc4752875452ee06a8f3211df70ad8fa'}}

## fit_frozen

원본: [05_model_optimize/Extra_Trees/Normal/scripts/fit_frozen.py](https://github.com/ryu980920/Machine_Learning/blob/ec30f7fd4a97de9f8336df2eef6a52b394c810c0/05_model_optimize/Extra_Trees/Normal/scripts/fit_frozen.py)  
Git blob: `60627ee546d0ac9be4b9e01d7e70f65a698edd7d`. [기록 및 결과](https://github.com/ryu980920/Machine_Learning/tree/ec30f7fd4a97de9f8336df2eef6a52b394c810c0/05_model_optimize/Extra_Trees/Normal/logs)

Fit one frozen Extra Trees candidate on the development training data.

CLI 인자 (정적 추출): `--mode`, `--out`, `--jobs`

In [ ]:
%%archive_source 05_model_optimize/Extra_Trees/Normal/scripts/fit_frozen.py
"""Fit one frozen Extra Trees candidate on the development training data."""
import argparse
import json
from pathlib import Path

import joblib
import numpy as np

from run_extra_trees import BASE, LABELS, WRITERS, Config, Study
from run_extra_trees import extract, process
from sklearn.ensemble import ExtraTreesClassifier


def predict(model, images, features):
    """Predict standardized 28 × 28 images with 0–255 pixel intensities."""
    array = np.asarray(images, dtype=np.float32)
    if array.ndim != 3 or array.shape[1:] != (28, 28) or len(array) == 0:
        raise ValueError("Expected a batch of standardized 28 × 28 images.")
    if not np.isfinite(array).all() or array.min() < 0 or array.max() > 255:
        raise ValueError("Expected 0–255 pixel intensities.")
    x = process(array.reshape(-1, 784), "com", 20)
    return model.predict(extract(x, features))


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--mode", choices=("normal", "combined"), required=True)
    parser.add_argument("--out", type=Path, required=True, help="New output directory")
    parser.add_argument("--jobs", type=int, default=4)
    args = parser.parse_args()
    manifest = BASE / "Combined" / "results" / "review" / "freeze.json"
    recipe = json.loads(manifest.read_text())[args.mode]
    config = Config(**recipe["config"])
    args.out.mkdir(parents=True, exist_ok=False)
    study = Study(args)
    study.audit()
    official, _, hands = study.inputs(config)
    y = study.y_train
    training_writers = recipe["training_writers"]
    if training_writers:
        if tuple(training_writers) != WRITERS:
            raise ValueError("Unexpected Combined writer set.")
        official = np.concatenate([official, *[hands[w] for w in training_writers]])
        y = np.concatenate([y, *[study.y_hand[w] for w in training_writers]])
    if len(y) != recipe["official_training_n"] + recipe["handmade_training_n"]:
        raise ValueError("Frozen training size mismatch.")

    model = ExtraTreesClassifier(n_estimators=config.n_estimators, max_features=config.max_features,
                                min_samples_leaf=config.min_samples_leaf, max_depth=config.max_depth,
                                random_state=recipe["seed"], n_jobs=args.jobs)
    model.fit(official, y)
    probe = study.official["X_train"][:16].reshape(-1, 28, 28)
    prediction = predict(model, probe, config.features)
    target = args.out / "extra_trees.joblib"
    joblib.dump({"model": model, "recipe": recipe, "labels": LABELS}, target, compress=3)
    restored = joblib.load(target)
    if not np.array_equal(prediction, predict(restored["model"], probe, config.features)):
        raise ValueError("Model predictions changed after serialization.")
    print(f"Fitted {args.mode} on {len(y):,} training images. Saved {target}.", flush=True)


if __name__ == "__main__":
    main()


## aggregate

원본: [05_model_optimize/Extra_Trees/Normal/scripts/aggregate.py](https://github.com/ryu980920/Machine_Learning/blob/ec30f7fd4a97de9f8336df2eef6a52b394c810c0/05_model_optimize/Extra_Trees/Normal/scripts/aggregate.py)  
Git blob: `14dab185b29c7e051ed8b542068781684ce4307b`. [기록 및 결과](https://github.com/ryu980920/Machine_Learning/tree/ec30f7fd4a97de9f8336df2eef6a52b394c810c0/05_model_optimize/Extra_Trees/Normal/logs)

Audit saved predictions and generate Extra Trees result tables and figures.

CLI 인자 (정적 추출): `--mode`, `--source`, `--out`

In [ ]:
%%archive_source 05_model_optimize/Extra_Trees/Normal/scripts/aggregate.py
"""Audit saved predictions and generate Extra Trees result tables and figures."""
import argparse
import csv
import json
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np

from run_extra_trees import (
    DATA, FOCUS, LABELS, PREPROCESSING, TUNING, WRITERS, choose,
    load_official, load_writers, official_average, score, write_csv,
)

METRICS = ("accuracy", "macro_precision", "macro_recall", "macro_f1",
           "digits_accuracy", "letters_accuracy")
BLUE, ORANGE, GREEN = "#1261a0", "#b84e1b", "#148471"
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False,
                     "axes.titleweight": "bold", "axes.labelcolor": "#243e51",
                     "font.size": 10, "savefig.facecolor": "white"})


def read_json(path):
    return json.loads(path.read_text())


def save(figure, path):
    figure.savefig(path, dpi=180, bbox_inches="tight", pad_inches=.15)
    plt.close(figure)


def load_and_verify(out, combined):
    official = load_official(DATA / "datasets/MNIST_EMNIST_official_split_112000.npz")
    y_val = official["y_val"]
    writers = load_writers(DATA / "Hand_made")
    y_hand = np.concatenate([writers[w]["y"] for w in WRITERS])
    runs, predictions = {}, {}
    for folder in sorted((out / "runs").iterdir()):
        if not (folder / "summary.json").exists():
            continue
        run = read_json(folder / "summary.json")
        name = run["name"]
        runs[name] = run
        with np.load(folder / "predictions.npz", allow_pickle=False) as z:
            data = dict(z)
        predictions[name] = data
        hand = np.concatenate([data[f"writer_{w}"] for w in WRITERS])
        if score(y_hand, hand) != run["hand"]:
            raise ValueError(f"Hand-made metrics mismatch in {name}")
        is_combined = combined and name != "CTRL_normal"
        expected_folds = {str(w) for w in WRITERS} if is_combined else {"all"}
        if set(run["official"]) != expected_folds or set(run["timing"]) != expected_folds:
            raise ValueError(f"Incorrect validation folds in {name}")
        for fold in expected_folds:
            times = run["timing"][fold]
            train_ids = [w for w in WRITERS if w != int(fold)] if is_combined else []
            if times["training_writers"] != train_ids or times["handmade_train_n"] != 640*len(train_ids):
                raise ValueError(f"Incorrect writer split in {name}/{fold}")
            if score(y_val, data[f"official_{fold}"]) != run["official"][fold]:
                raise ValueError(f"Official metrics mismatch in {name}/{fold}")
        for writer in WRITERS:
            if score(writers[writer]["y"], data[f"writer_{writer}"]) != run["writers"][str(writer)]:
                raise ValueError(f"Writer metrics mismatch in {name}/{writer}")
    return runs, predictions, writers


def plot_geometry(geometry, out):
    chosen = [row for row in geometry if row["dataset"] != "official_train"]
    names = ["Official val", "Writer 41", "Writer 42", "Writer 43", "Writer 44"]
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.2), layout="constrained")
    for ax, key, title in zip(axes, ("long_side", "ink_pixels"),
                             ("Bounding-box long side (px)", "Ink pixels at intensity ≥32")):
        x = np.arange(len(names))
        for offset, family, color in ((-.17, "digits", BLUE), (.17, "letters", ORANGE)):
            ax.bar(x+offset, [float(row[key]) for row in chosen if row["family"] == family],
                   .33, label=family.title(), color=color)
        ax.set(title=title, xticks=x, xticklabels=names)
        ax.tick_params(axis="x", labelrotation=20)
        ax.legend(frameon=False)
    save(fig, out / "input_geometry.png")


def plot_stages(preprocessing, tuning, selected, floor, out):
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.4), layout="constrained")
    for ax, metric, ylabel in zip(axes, ("official", "hand"),
                                  ("Official validation accuracy (%)", "Hand-made macro F1 (%)")):
        for i, group in enumerate((preprocessing, tuning)):
            index = np.arange(len(group)) + 7*i
            values = [official_average(row)*100 if metric == "official"
                      else row["hand"]["macro_f1"]*100 for row in group]
            ax.plot(index, values, marker="o", color=(BLUE, GREEN)[i], linewidth=1.4)
            for x, value, row in zip(index, values, group):
                if row["name"] == selected:
                    ax.scatter(x, value, s=95, facecolors="none", edgecolors=ORANGE,
                               linewidth=2, zorder=4)
        ax.set(xticks=[*range(6), *range(7, 13)],
               xticklabels=[r["name"][:3] for r in preprocessing+tuning], ylabel=ylabel)
        ax.tick_params(axis="x", labelrotation=40)
        ax.grid(axis="y", alpha=.25)
    axes[0].axhline(floor*100, color=ORANGE, linestyle="--", label="Eligibility floor")
    axes[0].legend(frameon=False)
    save(fig, out / "stage_comparison.png")


def plot_matrices(raw, final, out):
    pairs = [("Hand-made baseline", [raw["hand"]]), ("Hand-made selected", [final["hand"]]),
             ("Official validation selected", list(final["official"].values()))]
    fig, axes = plt.subplots(1, 3, figsize=(11, 3.7), layout="constrained")
    for ax, (title, rows) in zip(axes, pairs):
        counts = np.asarray([r["confusion_matrix"] for r in rows]).mean(axis=0)
        normalized = counts/counts.sum(axis=1, keepdims=True)
        im = ax.imshow(normalized, cmap="viridis", vmin=0, vmax=1)
        ax.set(title=title, xticks=range(16), yticks=range(16),
               xticklabels=list(LABELS), yticklabels=list(LABELS), xlabel="Predicted", ylabel="True")
        ax.tick_params(labelsize=7, length=0)
    fig.colorbar(im, ax=axes, shrink=.69, label="Row share")
    save(fig, out / "confusion_matrices.png")


def plot_errors(raw, final, writers, predictions, out):
    fig, ax = plt.subplots(figsize=(8.9, 3), layout="constrained")
    for offset, row, color, name in ((-.17, raw, ORANGE, "Baseline"), (.17, final, BLUE, "Selected")):
        ax.bar(np.arange(len(FOCUS))+offset, list(row["hand"]["focused_errors"].values()),
               .33, color=color, label=name)
    ax.set(xticks=np.arange(len(FOCUS)), xticklabels=[f"{s}→{d}" for s, d in FOCUS],
           ylabel="Wrong predictions (of 160 per class)")
    ax.legend(frameon=False)
    save(fig, out / "directional_errors.png")

    raw_pred, final_pred = (predictions[r["name"]] for r in (raw, final))
    fig, axes = plt.subplots(2, len(FOCUS), figsize=(10, 3), layout="constrained")
    examples = []
    for column, (source, dest) in enumerate(FOCUS):
        correct, wrong = LABELS.index(source), LABELS.index(dest)
        for row_id, outcome in enumerate(("corrected", "remains")):
            ax = axes[row_id, column]
            ax.set_axis_off()
            for w, entry in writers.items():
                before, after = raw_pred[f"writer_{w}"], final_pred[f"writer_{w}"]
                match = (entry["y"] == correct) & (before == wrong)
                match &= after == (correct if outcome == "corrected" else wrong)
                indices = np.flatnonzero(match)
                if indices.size:
                    i = indices[0]
                    ax.imshow(entry["X"][i].reshape(28, 28), cmap="gray", vmin=0, vmax=255)
                    ax.set_title(f"{source}→{dest}  W{w}", fontsize=8)
                    examples.append(dict(outcome=outcome, writer=w, row_index_zero_based=i,
                                         label=source, baseline_pred=dest,
                                         selected_pred=LABELS[after[i]]))
                    break
            else:
                ax.text(.5, .5, "—", ha="center", va="center", transform=ax.transAxes)
    fig.suptitle("Top: corrected examples   •   Bottom: errors retained")
    save(fig, out / "error_examples.png")
    if examples:
        write_csv(out / "error_examples.csv", examples)


def plot_curve(curve, out):
    fig, ax = plt.subplots(figsize=(8.7, 3.5), layout="constrained")
    x = [row["official_train_n"] for row in curve]
    for key, name, color in (("train_accuracy", "Training (resubstitution)", GREEN),
                             ("official_accuracy", "Official validation", BLUE),
                             ("hand_accuracy", "Hand-made development", ORANGE)):
        ax.plot(x, [row[key]*100 for row in curve], "-o", label=name, color=color)
    ax.set(xlabel="Official training images", ylabel="Accuracy (%)", xticks=x)
    ax.grid(alpha=.18)
    ax.legend(frameon=False)
    save(fig, out / "learning_curve.png")


def control(runs, predictions, writers, out):
    left, right = runs["CTRL_normal"], runs["R42_selected"]
    rows = []
    for w in WRITERS:
        key = str(w)
        y = writers[w]["y"]
        old = predictions[left["name"]][f"writer_{w}"] == y
        new = predictions[right["name"]][f"writer_{w}"] == y
        rows.append(dict(writer=w, n=len(y), normal_accuracy=old.mean(),
                         combined_accuracy=new.mean(), delta_accuracy=new.mean()-old.mean(),
                         normal_macro_f1=left["writers"][key]["macro_f1"],
                         combined_macro_f1=right["writers"][key]["macro_f1"],
                         corrected=int((~old & new).sum()), regressed=int((old & ~new).sum())))
    write_csv(out / "matched_writers.csv", rows)
    directions = []
    for key, old in left["hand"]["focused_errors"].items():
        new = right["hand"]["focused_errors"][key]
        directions.append(dict(direction=key, normal_errors=old, combined_errors=new,
                               difference=new-old, support=160))
    write_csv(out / "matched_directional_errors.csv", directions)
    fig, ax = plt.subplots(figsize=(7.8, 3.2), layout="constrained")
    for row in rows:
        y = row["writer"]
        a, b = row["normal_accuracy"]*100, row["combined_accuracy"]*100
        ax.plot([a, b], [y, y], color="#a9b6bf", linewidth=2)
        ax.scatter([a, b], [y, y], color=[ORANGE, BLUE])
    ax.scatter([], [], color=ORANGE, label="Official only")
    ax.scatter([], [], color=BLUE, label="+ 1,920 Hand-made")
    ax.set(yticks=WRITERS, xlabel="Held-out writer accuracy (%)", ylabel="Writer")
    ax.legend(frameon=False)
    save(fig, out / "matched_writers.png")


def aggregate(source, out, mode):
    out.mkdir(parents=True, exist_ok=True)
    combined = mode == "combined"
    runs, predictions, writers = load_and_verify(source, combined)
    selection = read_json(source / "selection.json")
    required = {*PREPROCESSING, *TUNING, "R42_selected", "R17_selected", "R73_selected",
                "LC20", "LC40", "LC60", "LC80"}
    if combined:
        required.add("CTRL_normal")
    if absent := required - runs.keys():
        raise ValueError(f"Missing runs: {absent}")
    raw, final = runs["P00_raw"], runs["R42_selected"]
    prep = [runs[name] for name in PREPROCESSING]
    tune = [runs[name] for name in TUNING]
    picked = choose(prep+tune, selection["official_accuracy_floor"])
    if picked["name"] != selection["selected"] or final["config"] != picked["config"]:
        raise ValueError("Selection could not be reproduced")
    for key, values in predictions[selection["selected"]].items():
        if not np.array_equal(predictions["R42_selected"][key], values):
            raise ValueError(f"Same-seed repeat differs: {key}")

    summary, folds, classes, matrix, pairs = [], [], [], [], []
    for name, r in runs.items():
        measurements = list(r["timing"].values())
        summary.append(dict(run=name, seed=r["seed"], source=r["training_source"],
                            fraction=r["fraction"], **r["config"],
                            **{f"official_{m}": official_average(r, m) for m in METRICS},
                            **{f"hand_{m}": r["hand"][m] for m in METRICS},
                            official_accuracy_fold_sd=np.std([x["accuracy"] for x in r["official"].values()],
                                                             ddof=1) if len(measurements)>1 else "",
                            fit_seconds_mean=np.mean([t["fit_seconds"] for t in measurements]),
                            val_predict_ms_image=np.mean([t["val_predict_seconds"] for t in measurements])*1000/19200,
                            hand_predict_ms_image=np.mean([t["hand_predict_seconds"]*1000/t["hand_eval_n"]
                                                           for t in measurements])))
        for fold, values in r["official"].items():
            folds.append(dict(run=name, evaluation="official_validation", fold=fold,
                              writers_train=" ".join(map(str, r["timing"][fold]["training_writers"])),
                              n=values["n"], **{m: values[m] for m in METRICS},
                              **{key: val for key, val in r["timing"][fold].items()
                                 if key not in ("training_writers", "train_accuracy")}))
        evaluations = [("handmade_pooled", "all", r["hand"])]
        evaluations.extend(("handmade_writer", w, values) for w, values in r["writers"].items())
        evaluations.extend(("official_validation", f, values) for f, values in r["official"].items())
        for name_of_set, fold, values in evaluations:
            scope = dict(run=name, evaluation=name_of_set, fold=fold)
            classes.extend(dict(**scope, **c) for c in values["classes"])
            for actual, counts in zip(LABELS, values["confusion_matrix"]):
                matrix.append(dict(**scope, actual=actual, **dict(zip(LABELS, counts))))
            for pair, n in values["focused_errors"].items():
                support = values["classes"][LABELS.index(pair[0])]["support"]
                pairs.append(dict(**scope, direction=pair, errors=n, support=support, rate=n/support))
    for filename, rows in (("experiment_summary.csv", summary), ("fold_metrics.csv", folds),
                           ("class_metrics.csv", classes), ("confusion_matrices.csv", matrix),
                           ("directional_errors.csv", pairs)):
        write_csv(out / filename, rows)
    curve = []
    for name in ("LC20", "LC40", "LC60", "LC80", "R42_selected"):
        r = runs[name]
        t = list(r["timing"].values())
        curve.append(dict(run=name, fraction=r["fraction"], official_train_n=t[0]["official_train_n"],
                          handmade_train_n=t[0]["handmade_train_n"],
                          train_accuracy=np.mean([x["train_accuracy"] for x in t]),
                          official_accuracy=official_average(r), hand_accuracy=r["hand"]["accuracy"],
                          hand_macro_f1=r["hand"]["macro_f1"],
                          fit_seconds_mean=np.mean([x["fit_seconds"] for x in t])))
    write_csv(out / "learning_curve.csv", curve)
    with (source / "input_geometry.csv").open() as f:
        geometry = list(csv.DictReader(f))
    plot_geometry(geometry, out)
    plot_stages(prep, tune, selection["selected"], selection["official_accuracy_floor"], out)
    plot_matrices(raw, final, out)
    plot_errors(raw, final, writers, predictions, out)
    plot_curve(curve, out)
    if combined:
        control(runs, predictions, writers, out)


if __name__ == "__main__":
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--mode", required=True, choices=("normal", "combined"))
    parser.add_argument("--source", required=True, type=Path)
    parser.add_argument("--out", required=True, type=Path)
    args = parser.parse_args()
    aggregate(args.source, args.out, args.mode)


## render_report

원본: [05_model_optimize/Extra_Trees/Normal/scripts/render_report.py](https://github.com/ryu980920/Machine_Learning/blob/ec30f7fd4a97de9f8336df2eef6a52b394c810c0/05_model_optimize/Extra_Trees/Normal/scripts/render_report.py)  
Git blob: `b92c5cc42c0134974d78907ecd2c3dc4e3794201`. [기록 및 결과](https://github.com/ryu980920/Machine_Learning/tree/ec30f7fd4a97de9f8336df2eef6a52b394c810c0/05_model_optimize/Extra_Trees/Normal/logs)

Render a Korean PDF from measured Extra Trees results and generated charts.

CLI 인자 (정적 추출): `--mode`, `--source`, `--results`, `--output`, `--review-dir`

In [ ]:
%%archive_source 05_model_optimize/Extra_Trees/Normal/scripts/render_report.py
"""Render a Korean PDF from measured Extra Trees results and generated charts."""
import argparse
import csv
import json
from html import escape
from pathlib import Path

import numpy as np
from weasyprint import HTML

from run_extra_trees import LABELS, PREPROCESSING, TUNING, WRITERS, choose, official_average

STYLE = """
@page {
  size: A4;
  margin: 14mm 15mm 17mm;
  border-top: 3pt solid #1261a0;
  padding-top: 4mm;
  @bottom-left { content: string(report-label); font: 8pt sans-serif; color: #496575; }
  @bottom-right { content: counter(page, decimal-leading-zero); font: 8pt sans-serif; color: #496575; }
}
html {font-family:"Noto Sans CJK KR",sans-serif;}
body {font-size:10pt;color:#172f41;line-height:1.41;padding:0;margin:0;}
article {break-after:page;}
article:last-child {break-after:auto;}
.report-label {string-set:report-label content();font-size:0;height:0;overflow:hidden;}
h1 {font-size:30pt;color:#0c3656;line-height:1.20;margin:4pt 0 12pt;}
h2 {font-size:22pt;color:#0c3656;line-height:1.25;margin:4pt 0 12pt;}
h3 {font-size:13pt;color:#1261a0;margin:15pt 0 5pt;}
p {margin:0 0 9pt;}
.kicker {color:#1261a0;font-size:9pt;font-weight:bold;margin-bottom:6pt;}
.note {font-size:9pt;color:#3c576a;}
.lead {font-size:12pt;margin-bottom:14pt;}
table {width:100%;border-collapse:collapse;margin:7pt 0 12pt;}
th {font-weight:bold;background:#e6eff5;text-align:left;color:#103b57;}
td,th {border-bottom:1px solid #d9e2ea;padding:5pt 6pt;font-size:8.8pt;}
img {display:block;width:100%;height:auto;max-height:65mm;object-fit:contain;}
.errors img {max-height:46mm;}
.closing img {max-height:47mm;}
.review-errors img {max-height:28mm;}
.review-errors td,.review-errors th {padding:3pt 6pt;}

"""

PREP_TEXT = {
    "P00_raw": "원본 픽셀",
    "P01_com": "CoM 정렬",
    "P02_size": "CoM + 크기",
    "P03_stroke": "CoM + 크기·획",
    "P04_hybrid": "크기·획 + 픽셀·HOG",
    "P05_hog": "크기·획 + HOG 단독",
}
TUNE_TEXT = {
    "T01_feat08": "max_features = 0.08",
    "T02_feat18": "max_features = 0.18",
    "T03_leaf2": "min_samples_leaf = 2",
    "T04_depth24": "max_depth = 24",
    "T05_trees64": "n_estimators = 64",
    "T06_trees160": "n_estimators = 160",
}


def paragraph(value, style=""):
    return f'<p class="{style}">{escape(str(value))}</p>'


def title(kicker, heading, first=False):
    tag = "h1" if first else "h2"
    return paragraph(kicker, "kicker") + f"<{tag}>{escape(heading)}</{tag}>"


def sub(heading):
    return f"<h3>{escape(heading)}</h3>"


def table(headers, rows):
    head = "".join(f"<th>{escape(str(v))}</th>" for v in headers)
    lines = "".join("<tr>"+"".join(f"<td>{escape(str(v))}</td>" for v in row)+"</tr>" for row in rows)
    return f"<table><tr>{head}</tr>{lines}</table>"


def figure(name, caption):
    return f'<img src="{name}.png" alt=""/>' + paragraph(caption, "note")


def pct(value):
    return f"{100*value:.2f}%"


def delta(value):
    return f"{100*value:+.2f}%p"


def read_csv(path):
    with path.open() as stream:
        return list(csv.DictReader(stream))


def average(rows, function):
    return float(np.mean([function(row) for row in rows]))


def seed_stats(seeds, function):
    values = [function(r) for r in seeds]
    return f"{pct(np.mean(values))} ± {100*np.std(values, ddof=1):.2f}%p"


def pages(out, results, mode, review=None):
    comb = mode == "Combined"
    runs = {p.parent.name: json.loads(p.read_text()) for p in (out / "runs").glob("*/summary.json")}
    selection = json.loads((out / "selection.json").read_text())
    audit = json.loads((out / "data_audit.json").read_text())
    env = json.loads((out / "environment.json").read_text())
    baseline, final = runs["P00_raw"], runs["R42_selected"]
    seeds = [runs[f"R{s}_selected"] for s in (42, 17, 73)]
    sel, config = selection["selected"], final["config"]
    hand = lambda r: r["hand"]["accuracy"]
    description = PREP_TEXT.get(selection["preprocessing"])
    final_method = (PREP_TEXT.get(sel) or description) + (f" · {TUNE_TEXT[sel]}" if sel in TUNE_TEXT else "")
    count = sum(len(r["official"]) for r in runs.values())
    hand_kind = "작성자 보류 OOF" if comb else "팀 손글씨 개발 평가"
    split = "공식 Train 76,800 + 손글씨 1,920장/Fold" if comb else "공식 Train 76,800장"
    summary = title("머신러닝 · 16클래스 손글씨 분류", f"Extra Trees {mode}", True)
    summary += paragraph(f"{hand_kind} 정확도가 {pct(hand(baseline))}에서 "
                         f"{pct(hand(final))}로 증가했다. 공식 Validation은 "
                         f"{pct(official_average(baseline))}에서 {pct(official_average(final))}로 변했다.", "lead")
    summary += table(("평가 지표", "원본 픽셀 · 시드 42", "선정 · 시드 42", "선정 · 3시드 평균"),
                     ((label, pct(fn(baseline)), pct(fn(final)), seed_stats(seeds, fn)) for label, fn in (
                         ("공식 Accuracy", official_average),
                         ("공식 Macro F1", lambda r: official_average(r, "macro_f1")),
                         ("공식 Macro Precision", lambda r: official_average(r, "macro_precision")),
                         ("공식 Macro Recall", lambda r: official_average(r, "macro_recall")),
                         ("손글씨 Accuracy", hand),
                         ("손글씨 Macro Precision", lambda r: r["hand"]["macro_precision"]),
                         ("손글씨 Macro Recall", lambda r: r["hand"]["macro_recall"]),
                         ("손글씨 Macro F1", lambda r: r["hand"]["macro_f1"]),
                     )))
    summary += sub("선정 구성") + paragraph(f"{sel} · {final_method} · {config['n_estimators']}개 트리. "
                                               f"학습 구성은 {split}이다.")
    if comb:
        left = runs["CTRL_normal"]
        summary += paragraph(f"선정 설정을 고정한 대조에서 손글씨 1,920장을 추가하자 "
                             f"OOF와 동일한 2,560장의 정확도가 {pct(hand(left))}에서 "
                             f"{pct(hand(final))}로 변했다({delta(hand(final)-hand(left))}).")
    summary += sub("기존 팀 보고서와의 비교")
    comparators = (("MLP Combined", "97.08%", "92.36%"),
                   ("HGB Combined", "98.26%", "93.39%")) if comb else (
                   ("MLP Normal", "96.54%", "92.07%"),
                   ("HGB Normal", "98.28%", "91.43%"))
    summary += table(("보고서 최종 후보", "공식 Validation", "손글씨 개발"),
                     [(f"Extra Trees {mode}", pct(average(seeds, official_average)), pct(average(seeds, hand))),
                      *comparators])
    summary += paragraph("MLP·HGB 수치는 각 보고서의 최종 3시드 평균이다. "
                         "Extra Trees 역시 시드별 점수를 평균했으며 ±값은 표본 표준편차다. 보고서마다 시드, 후보 선정 기준과 실행 환경은 다르다. "
                         "Original Test 16,000장과 교수님 Test는 이 연구에서 평가하지 않았다.", "note")
    if comb and review and (review / "paired_controls.csv").is_file():
        summary += paragraph("후속 대조의 시드별 결과와 독립 평가용 후보 결정은 마지막 쪽에 정리했다.", "note")
    yield summary

    data = title("데이터 감사와 기준선", "공식 자료와 작성자별 분리")
    data += table(("자료", "표본 수", "사용 방식"),
                  [("공식 Train", "76,800", "모든 실험의 학습 원천"),
                   ("공식 Validation", "19,200", "동일 분할로 후보를 비교"),
                   ("41·42·43·44번", "각 640", "작성자별 16클래스 × 40장"),
                   ("Fold 학습 손글씨" if comb else "손글씨 개발 평가",
                    "1,920 / Fold" if comb else "2,560",
                    "보류 작성자 제외" if comb else "공식 Train으로 학습한 모델의 예측")])
    data += paragraph("공식 원천 96,000장은 StratifiedGroupKFold 5분할의 첫 Fold를 사용했다. "
                      "원본과 공식 증강본의 source group을 함께 묶었다. 팀 보존 인덱스·그룹 ID와 완전히 일치했고 "
                      "Train–Validation 그룹 교집합은 0이었다. 다섯 NPZ의 SHA-256은 팀 기록과 일치했다. "
                      "작성자 간 및 공식 자료와 손글씨 간 동일 원본 픽셀 배열 교집합도 0이었다.")
    if comb:
        data += paragraph("Fold마다 공식 Train과 다른 작성자 세 명의 데이터로 새 모델을 학습했다. "
                          "보류 손글씨 예측 640장씩을 모아 OOF를 계산했다. 공식 Validation은 "
                          "같은 19,200장에 대한 Fold별 지표의 평균이다.")
    else:
        data += paragraph("P00의 공식 Validation 96.96875%와 Macro F1 0.96954는 W02 실험 2의 "
                          "공표값 96.97%와 0.9695를 재현했다. 최신 팀 손글씨 정확도 61.60%도 기존 기록과 일치했다.")
    data += figure("input_geometry", "획 픽셀은 밝기 32 이상을 세었다. 공식 문자의 긴 변은 숫자보다 3.91 px 길다. "
                   "팀 손글씨의 숫자·문자 크기는 작성자별로 비슷하다.")
    data += paragraph("작성자 41번의 숫자 획 픽셀 수는 평균 86.0개로 공식 Validation 숫자 134.0개의 64.2%였다. "
                      "이 분포 차이를 크기와 획 혼합 대조의 가설로 사용했다.")
    data += paragraph(f"입력 파일 해시·레이블 분포·분할 근거: data_audit.json, official_split_indices.npz. "
                      f"기준 소스 커밋 {audit['repository_commit'][:12]}.", "note")
    yield data

    prep = title("전처리와 특징 비교", "CoM·크기·획·HOG")
    rows = []
    for name in PREPROCESSING:
        r = runs[name]
        rows.append((name[:3], PREP_TEXT[name], pct(official_average(r)), pct(hand(r)),
                     f"{r['hand']['macro_f1']:.4f}",
                     "후속 탐색" if name == selection["preprocessing"] else
                     "기준 미달" if official_average(r)<selection["official_accuracy_floor"] else "비교군"))
    prep += table(("ID", "변경 조건", "공식 Acc.", "손글씨 Acc.", "손글씨 F1", "판정"), rows)
    for a, b, label in (("P00_raw", "P01_com", "CoM 효과"),
                        ("P01_com", "P02_size", "크기 증강 효과"),
                        ("P02_size", "P03_stroke", "획 혼합 효과"),
                        ("P03_stroke", "P04_hybrid", "HOG 추가 효과"),
                        ("P04_hybrid", "P05_hog", "혼합 특징의 픽셀 제거")):
        l, r = runs[a], runs[b]
        prep += paragraph(f"{label} · 공식 {delta(official_average(r)-official_average(l))}, "
                          f"손글씨 {delta(hand(r)-hand(l))}. 손글씨 Macro F1 "
                          f"{l['hand']['macro_f1']:.4f} → {r['hand']['macro_f1']:.4f}.")
    aug = runs["P03_stroke"]["preprocessing"]["augmentation"]
    prep += paragraph(f"CoM은 픽셀 밝기 무게중심을 중앙에 옮긴다. 크기 조건은 공식 Train {aug['n_aug']:,}장을 "
                      "긴 변 16·18·20·22·24 px에 클래스별 균형 배정해 교체했다. "
                      f"획 조건은 침식·팽창 혼합 0.28을 추가해 {aug['n_stroke_changed']:,}장에 적용했고 "
                      f"위상이 바뀐 {aug['n_stroke_rejected']:,}장은 혼합을 취소했다.")
    prep += paragraph("HOG는 9방향, 7×7 셀, 2×2 셀 블록, L2-Hys의 324차원이다. "
                      "혼합 특징은 원 픽셀 784개와 합쳐 1,108차원이다. 모든 전처리 비교에서 "
                      "트리 100개와 기본 분기 설정을 사용했다. 크기·획 증강은 공식 Train에만 적용했다. CoM은 밝기 32 이상 획의 무게중심을 (13.5, 13.5)로 이동하며 정수 이동 후 0으로 채운다.")
    yield prep

    tune = title("Extra Trees 튜닝", "공식 검증과 손글씨의 절충")
    tune += figure("stage_comparison", "P군은 전처리·특징을 비교했다. T군은 선택한 P 설정을 기준으로 한 변수만 바꿨다. "
                   "주황 점선은 공식 Validation 자격선이며 원형 표시는 최종 선택 조건이다.")
    tune += table(("실험", "변경 변수", "공식 Acc.", "손글씨 Acc.", "손글씨 F1", "학습/모델"),
                  [(n[:3], TUNE_TEXT[n], pct(official_average(runs[n])), pct(hand(runs[n])),
                    f"{runs[n]['hand']['macro_f1']:.4f}",
                    f"{average(list(runs[n]['timing'].values()), lambda z: z['fit_seconds']):.1f}s")
                   for n in TUNING])
    tune += paragraph(f"자격선은 P00 기준선 −0.50%p에서 정한 {pct(selection['official_accuracy_floor'])}였다. "
                      "이를 충족한 후보 중 손글씨 Macro F1, 손글씨 Accuracy, 공식 Accuracy, 적은 트리 수 순으로 "
                      f"선정했다. 최종 {sel[:3]}의 공식 Accuracy는 {pct(official_average(final))}, "
                      f"손글씨 Macro F1은 {final['hand']['macro_f1']:.4f}였다.")
    ranked = [runs[n] for n in (*PREPROCESSING, *TUNING) if n != sel]
    runner_up = choose(ranked, selection["official_accuracy_floor"])
    top_time = average(list(runs[sel]["timing"].values()), lambda row: row["fit_seconds"])
    next_time = average(list(runner_up["timing"].values()), lambda row: row["fit_seconds"])
    tied = hand(final) == hand(runner_up)
    comparison = (f"두 조건의 손글씨 정확도는 {pct(hand(final))}로 같았다." if tied else
                  f"손글씨 정확도 차이는 {delta(hand(final)-hand(runner_up))}였다.")
    tune += paragraph(f"차순위 {runner_up['name'][:3]} 대비 손글씨 Macro F1 차이는 "
                      f"{final['hand']['macro_f1']-runner_up['hand']['macro_f1']:+.5f}였다. "
                      f"{comparison} 탐색 중 평균 적합 시간은 차순위 {next_time:.1f}초, "
                      f"선정 조건 {top_time:.1f}초였다.")
    tune += paragraph(f"최종 설정 · {config['n_estimators']} trees, max_features={config['max_features']}, "
                      f"min_samples_leaf={config['min_samples_leaf']}, max_depth={config['max_depth']}, bootstrap=False. "
                      "결과는 비교한 일변수 범위에 해당하며 하이퍼파라미터 조합의 전수 탐색은 수행하지 않았다.", "note")
    yield tune

    errors = title("클래스별 오류 분석", "유사 글자의 오분류 방향")
    errors += figure("confusion_matrices", "행은 정답, 열은 예측이다. 공식 Validation은 "
                     + ("Fold별 같은 행의 정규화 행렬을 평균했다." if comb else "시드 42 모델의 행 정규화 행렬이다."))
    errors += figure("directional_errors", "팀 손글씨는 정답 클래스마다 160장이다. 막대 높이는 오답 건수다.")
    before, after = baseline["hand"]["focused_errors"], final["hand"]["focused_errors"]
    errors += table(("방향", *(k.replace("->", "→") for k in before)),
                    [("기준", *before.values()), ("선정", *after.values())])
    changed = sorted(before, key=lambda k: after[k]-before[k])
    best, weakest = changed[0], changed[-1]
    errors += paragraph(f"{best.replace('->', '→')} {before[best]}→{after[best]}건으로 가장 크게 줄었다. "
                        f"{weakest.replace('->', '→')} {before[weakest]}→{after[weakest]}건은 여전히 확인할 오류다.")
    worst = sorted(final["hand"]["classes"], key=lambda row: row["recall"])[:3]
    errors += paragraph("최종 손글씨에서 Recall이 가장 낮은 클래스는 "
                        + ", ".join(f"{r['label']} {pct(r['recall'])} ({r['errors']}건 오류)" for r in worst)
                        + "였다. 방향별 건수 감소와 클래스별 Recall은 CSV에서 함께 확인할 수 있다.")
    errors += figure("error_examples", "원본 손글씨의 행 인덱스를 별도 CSV에 기록했다. "
                     "윗줄은 동일 방향 오답이 정답으로 개선된 사례, 아랫줄은 같은 방향 오답이 남은 사례다.")
    yield errors

    learning = title("학습 규모와 실행 시간", "데이터 수 기반 Learning Curve")
    learning += figure("learning_curve", "시드 42. 공식 Train의 클래스 층화 중첩 부분집합을 사용했다. "
                       "훈련 정확도는 적합에 사용한 표본에 대한 값이다. "
                       + ("Combined는 모든 점과 Fold에서 손글씨 1,920장을 고정했다." if comb else "손글씨는 평가에만 사용했다."))
    curve = read_csv(results / "learning_curve.csv")
    learning += table(("공식 Train", "손글씨 Train", "Train Acc.", "공식 Val.", "손글씨 Acc.", "적합 시간"),
                      [(f"{int(r['official_train_n']):,}", f"{int(r['handmade_train_n']):,}",
                        pct(float(r["train_accuracy"])), pct(float(r["official_accuracy"])),
                        pct(float(r["hand_accuracy"])), f"{float(r['fit_seconds_mean']):.1f}s") for r in curve])
    first, last = curve[0], curve[-1]
    peak = max(curve, key=lambda row: float(row["hand_accuracy"]))
    learning += paragraph(f"공식 학습량을 20%에서 100%로 늘리자 공식 Validation은 "
                          f"{delta(float(last['official_accuracy'])-float(first['official_accuracy']))}, "
                          f"손글씨 정확도는 {delta(float(last['hand_accuracy'])-float(first['hand_accuracy']))} 변했다.")
    if comb:
        at_sixty = next(row for row in curve if row["run"] == "LC60")
        hand_share = lambda row: int(row["handmade_train_n"]) / (
            int(row["official_train_n"]) + int(row["handmade_train_n"]))
        learning += paragraph(
            f"46,079장과 61,440장에서 손글씨 정확도 {pct(float(at_sixty['hand_accuracy']))}가 공동 최고였다. "
            f"76,800장의 공식 Validation은 46,079장 대비 "
            f"{delta(float(last['official_accuracy'])-float(at_sixty['official_accuracy']))}, "
            f"손글씨는 {delta(float(last['hand_accuracy'])-float(at_sixty['hand_accuracy']))}였다. "
            f"손글씨 학습 비중은 {pct(hand_share(at_sixty))}에서 {pct(hand_share(last))}로 줄었다. "
            "시드 42에서 관측한 결과이며 공식 학습량과 손글씨 비중이 함께 변했다.")
    else:
        learning += paragraph(
            f"손글씨 최고 정확도는 공식 학습 {int(peak['official_train_n']):,}장에서 "
            f"{pct(float(peak['hand_accuracy']))}였다.")
    times = list(final["timing"].values())
    avg_fit = average(times, lambda r: r["fit_seconds"])
    avg_val = average(times, lambda r: r["val_predict_seconds"]*1000/19200)
    avg_hand = average(times, lambda r: r["hand_predict_seconds"]*1000/r["hand_eval_n"])
    learning += paragraph(f"시드 42 선정 모델의 평균 적합 시간은 모델당 {avg_fit:.2f}초였다. "
                          f"배치 추론은 공식 Validation 장당 {avg_val:.3f} ms, 손글씨 장당 {avg_hand:.3f} ms였다. "
                          "특징 추출과 입력 변환은 모델 추론 시간에서 제외했다. "
                          "Normal과 Combined가 같은 CPU에서 일부 병행 실행돼 시간은 해당 실행의 관측값이다.")
    learning += paragraph(f"Python {env['python']}, scikit-learn {env['packages']['scikit-learn']}, "
                          f"{env['cpu']}, 모델당 n_jobs={env['jobs']}. "
                          f"완료한 적합은 총 {count}회이며 반복과 학습곡선 적합을 포함한다.", "note")
    yield learning

    closing = title("통제 실험과 결론" if comb else "작성자별 진단과 결론",
                    "손글씨 추가 학습 효과" if comb else "최종 개발 후보")
    if comb:
        matched = read_csv(results / "matched_writers.csv")
        closing += figure("matched_writers", "Combined 선정 조건·시드 42를 고정했다. "
                          "공식 Train만 사용한 모델과 작성자 3명 손글씨를 추가한 Fold 모델이 같은 보류 작성자를 예측했다.")
        closing += table(("작성자", "공식만 학습", "손글씨 추가", "차이", "개선 / 악화 건수"),
                         [(row["writer"], pct(float(row["normal_accuracy"])), pct(float(row["combined_accuracy"])),
                           delta(float(row["delta_accuracy"])), f"{row['corrected']} / {row['regressed']}")
                          for row in matched])
        left = runs["CTRL_normal"]
        weakest = min(matched, key=lambda row: float(row["delta_accuracy"]))
        closing += paragraph(f"작성자 간 변화 폭이 달랐다. 변화 폭이 가장 낮은 {weakest['writer']}번은 "
                             f"{pct(float(weakest['normal_accuracy']))} → {pct(float(weakest['combined_accuracy']))}였다. "
                             f"D→0은 {left['hand']['focused_errors']['D->0']}→{final['hand']['focused_errors']['D->0']}건, "
                             f"B→8은 {left['hand']['focused_errors']['B->8']}→{final['hand']['focused_errors']['B->8']}건이었다.")
        closing += paragraph(f"통제 비교의 정확도 차이는 {delta(hand(final)-hand(left))}, 공식 검증 차이는 "
                             f"{delta(official_average(final)-official_average(left))}였다. "
                             "작성자 4명을 대상으로 한 관측 차이이며 모집단 효과의 신뢰구간으로 해석할 수 없다.")
    else:
        closing += table(("작성자", "기준 Acc.", "선정 Acc.", "선정 Macro F1"),
                         [(w, pct(baseline["writers"][str(w)]["accuracy"]),
                           pct(final["writers"][str(w)]["accuracy"]),
                           f"{final['writers'][str(w)]['macro_f1']:.4f}") for w in WRITERS])
        closing += paragraph(f"최종 손글씨 정확도는 기준 대비 {delta(hand(final)-hand(baseline))}였다. "
                             "세 시드 반복은 추정기 초기 난수에 대한 안정성을 확인했다. "
                             "각 시드에서 동일한 네 작성자를 평가했다.")
    closing += sub("선정 범위") + paragraph("후보 탐색에 같은 공식 Validation과 팀 손글씨를 반복 사용했다. "
                        "따라서 성능 수치는 개발 결과이며 독립 작성자나 최종 Test의 정확도는 추정하지 않았다. "
                        "시드 평균은 개별 모델 점수의 평균이다. 교수님 Test와 공식 Original Test 평가는 미실시다.")
    freeze = review / "freeze.json" if review else None
    if freeze and freeze.is_file():
        recipe = json.loads(freeze.read_text()).get(mode.lower())
        if recipe:
            closing += sub("독립 평가용 후보 구성")
            closing += paragraph(
                f"{recipe['candidate']} · 시드 {recipe['seed']} · 공식 Train "
                f"{recipe['official_training_n']:,}장 · 손글씨 {recipe['handmade_training_n']:,}장. "
                "Extra Trees 한 모델을 학습하고 추론에는 CoM 정렬과 픽셀·HOG를 적용한다. "
                "손글씨 개발 OOF는 후보 비교에 사용한 Fold별 점수다.")
    closing += sub("비교 근거") + paragraph("팀 저장소 README·RESEARCH_PLAN·W02 실험 2/3 및 02_data 검증 문서. "
                    "팀 선행 보고서 MLP_Normal(19p), MLP_Combined(18p), HGB_Normal(23p), HGB_Combined(15p). "
                    "실험별 원자료는 runs/summary.json 및 predictions.npz, 통합 결과는 results의 CSV를 참조한다.", "note")
    yield closing

    if review:
        yield paired_error_page(review, mode)
        if comb and (review / "paired_controls.csv").is_file():
            yield controls_page(review)


def paired_error_page(review, mode):
    rows = [r for r in read_csv(review / "seven_to_nine_by_writer.csv") if r["mode"] == mode]
    totals = {key: sum(int(r[key]) for r in rows) for key in rows[0] if key not in ("mode", "writer")}
    output = title("동일 샘플의 예측 대조", "7→9 오류와 작성자별 차이")
    output += table(("작성자", "7 정답 기존 → 선정", "7→9 기존 → 선정", "기존 정답 → 9", "기존 기타 오답 → 9"),
                    [(r["writer"], f"{r['correct_before']} → {r['correct_after']}",
                      f"{r['before']} → {r['after']}", r["introduced_from_correct"],
                      r["introduced_from_other_error"]) for r in rows])
    output += paragraph(
        f"전체 숫자 7 {totals['true_sevens']}장 중 정답은 {totals['correct_before']}→"
        f"{totals['correct_after']}건, 7→9 오답은 {totals['before']}→{totals['after']}건이었다. "
        f"새 7→9 오류에는 기존 정답 {totals['introduced_from_correct']}건과 "
        f"기존의 다른 오답 {totals['introduced_from_other_error']}건이 포함됐다. "
        f"기존 7→9 가운데 {totals['resolved_to_correct']}건은 정답으로 바뀌었다.")
    output += paragraph("44번에서 7→9 오류가 가장 크게 늘었다. 전체 숫자 7의 인식률은 개선됐고 "
                        "잔존 오류는 9에 더 집중됐다. 여러 전처리와 트리 설정이 함께 바뀐 기준선 비교이므로 "
                        "개별 설정이 7→9 오류에 기여한 크기는 분리해 추정하지 않았다.")
    if mode == "Combined":
        classes = read_csv(review / "writer41_by_class.csv")
        changed = [r for r in classes if r["errors_before"] != r["errors_after"]]
        output += sub("41번에서 손글씨 추가 학습의 영향")
        output += table(("정답", "오류 기존 → 추가", "오답 → 정답", "정답 → 오답"),
                        [(r["actual"], f"{r['errors_before']} → {r['errors_after']}", r["corrected"],
                          r["regressed"]) for r in changed])
        output += paragraph("같은 설정의 공식 자료 단독 모델 대비 15건이 개선되고 21건이 악화됐다. "
                            "순오류는 숫자 6에서 4건, 숫자 9에서 3건 증가했다. "
                            "6에서 새로 틀린 4건은 C 2건, B와 5 각 1건이었다. D의 오류는 3건 줄었다. "
                            "41번의 획 밀도가 낮았다는 입력 특성은 관찰값이며, "
                            "획 굵기와 정확도 하락의 인과관계를 이 비교로 추정하지 않았다.")
    gallery = review / f"seven_to_nine_{mode.lower()}.png"
    if gallery.is_file():
        output += figure(gallery.resolve().as_uri().removesuffix(".png"),
                         "모두 정답 7. 왼쪽 4장은 새 7→9 오류, 오른쪽 4장은 9에서 7로 교정된 예시다. "
                         "위 숫자는 작성자와 원본 행 번호, 아래는 예측의 변화다.")
        output += paragraph("윗획 왼쪽이 아래로 꺾인 7이 새 오류와 교정 사례 모두에 보였다. "
                            "같은 형태에서도 예측이 달라져 이 시각적 특징만으로 오류 증가를 설명하기 어렵다.",
                            "note")
    output += paragraph("예측이 바뀐 원본 행 번호는 paired_error_examples.csv에 기록했다.", "note")
    return output


def controls_page(review):
    rows = read_csv(review / "paired_controls.csv")
    output = title("후속 대조 실험 · 같은 작성자 분할과 시드", "획 혼합과 분기 후보 수")
    for arm, heading in (("stroke00", "픽셀·HOG에서 획 혼합 0.00과 0.28"),
                         ("features08", "분기 특징 비율 0.08과 0.18")):
        control = [r for r in rows if r["comparison"] == arm and r["arm"] == arm]
        reference = [r for r in rows if r["comparison"] == arm and r["arm"] == "T02_reference"]
        score_delta = lambda key: [float(b[key])-float(a[key]) for a, b in zip(control, reference)]
        output += sub(heading)
        output += table(("시드", "공식 Acc. 차이", "손글씨 Acc. 차이", "손글씨 Macro F1 차이"),
                        [(a["seed"], delta(float(b["official_accuracy"])-float(a["official_accuracy"])),
                          delta(float(b["hand_accuracy"])-float(a["hand_accuracy"])),
                          f"{float(b['hand_macro_f1'])-float(a['hand_macro_f1']):+.5f}")
                         for a, b in zip(control, reference)])
        output += paragraph(
            f"차이는 T02에서 대조군을 뺀 값이다. 3시드 평균 차이는 공식 "
            f"{delta(np.mean(score_delta('official_accuracy')))}, 손글씨 "
            f"{delta(np.mean(score_delta('hand_accuracy')))}, "
            f"Macro F1 {np.mean(score_delta('hand_macro_f1')):+.5f}였다.")
    output += paragraph(
        "획 혼합 대조는 크기 증강 30%와 픽셀·HOG를 고정했다. 0.28은 획 혼합의 강도다. "
        "픽셀 단독 단계의 대조는 트리 설정이 달라 HOG와 획의 상호작용 크기를 직접 추정할 수 없다.")
    old_fit = average(control, lambda row: float(row["fit_seconds_mean"]))
    reference_fit = average(reference, lambda row: float(row["fit_seconds_mean"]))
    output += paragraph(
        "1,108개 입력 특징 중 각 분기에서 고려하는 기본 특징 수는 0.08에서 88개, 0.18에서 199개다. "
        "후보 수를 늘리면 무작위 분기에서 더 많은 변수를 비교하므로 계산량이 커지고 트리 간 다양성에도 영향을 줄 수 있다. "
        f"3시드에서 모델당 평균 적합 시간은 각각 {old_fit:.1f}초, {reference_fit:.1f}초였다. "
        "실행 시점과 CPU 부하가 달라 시간은 관측값이다.")
    freeze = review / "freeze.json"
    if freeze.is_file():
        decision = json.loads(freeze.read_text())["combined"]["report_decision"]
        output += paragraph(decision, "note")
    return output


def render(source, results, output, mode, review=None):
    report_pages = list(pages(source, results, mode, review))
    section_styles = {4: "errors", 6: "closing", 7: "review-errors"}
    articles = "".join(f'<article class="{section_styles.get(i, "")}">{body}</article>'
                       for i, body in enumerate(report_pages))
    date = "2026-11-23" if review else "2026-11-22"
    label = f"EXTRA TREES / {mode.upper()}  |  {date}"
    html = (f'<!doctype html><html lang="ko"><head><meta charset="utf-8">'
            f'<title>Extra Trees {mode} Optimization</title><style>{STYLE}</style></head>'
            f'<body><div class="report-label">{label}</div><main>{articles}</main></body></html>')
    document = HTML(string=html, base_url=str(results.resolve()) + "/").render()
    if len(document.pages) != len(report_pages):
        raise ValueError(f"Layout overflow: {len(document.pages)} pages for {len(report_pages)} sections")
    document.write_pdf(output)
    print(f"Rendered {len(document.pages)} pages: {output}", flush=True)


if __name__ == "__main__":
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--mode", choices=("Normal", "Combined"), required=True)
    parser.add_argument("--source", type=Path, required=True)
    parser.add_argument("--results", type=Path, required=True)
    parser.add_argument("--output", type=Path, required=True)
    parser.add_argument("--review-dir", type=Path)
    args = parser.parse_args()
    render(args.source, args.results, args.output, args.mode, args.review_dir)


## write_log

원본: [05_model_optimize/Extra_Trees/Normal/scripts/write_log.py](https://github.com/ryu980920/Machine_Learning/blob/ec30f7fd4a97de9f8336df2eef6a52b394c810c0/05_model_optimize/Extra_Trees/Normal/scripts/write_log.py)  
Git blob: `65deaa3ddc4752875452ee06a8f3211df70ad8fa`. [기록 및 결과](https://github.com/ryu980920/Machine_Learning/tree/ec30f7fd4a97de9f8336df2eef6a52b394c810c0/05_model_optimize/Extra_Trees/Normal/logs)

Write the measured experiment journal without duplicating result entry.

CLI 인자 (정적 추출): `--mode`, `--source`, `--output`

In [ ]:
%%archive_source 05_model_optimize/Extra_Trees/Normal/scripts/write_log.py
"""Write the measured experiment journal without duplicating result entry."""
import argparse
import json
from pathlib import Path

from run_extra_trees import PREPROCESSING, TUNING, official_average


HYPOTHESES = {
    "P00_raw": "W02 기본 픽셀 조건을 재현한다.",
    "P01_com": "중심 이동으로 위치 편차를 줄인다.",
    "P02_size": "공식 학습의 클래스별 크기 단서를 약화한다.",
    "P03_stroke": "침식·팽창 혼합으로 획 굵기 편차를 줄인다.",
    "P04_hybrid": "국소 방향을 픽셀과 함께 사용한다.",
    "P05_hog": "HOG가 픽셀 정보를 단독으로 대체할 수 있는지 본다.",
    "T01_feat08": "후보 분기 특징을 8%로 높이면 더 좋은 분기를 찾는다.",
    "T02_feat18": "후보 분기 특징을 18%로 더 높이면 검증 성능이 개선된다.",
    "T03_leaf2": "최소 잎 표본 수 2개로 과도한 분기를 제한한다.",
    "T04_depth24": "최대 깊이 24로 트리를 제한한다.",
    "T05_trees64": "트리 64개로 줄여도 성능이 유지된다.",
    "T06_trees160": "트리 160개로 늘리면 예측 분산이 감소한다.",
}


def write_log(source, output, mode):
    runs = {f.parent.name: json.loads(f.read_text()) for f in (source/"runs").glob("*/summary.json")}
    selection = json.loads((source/"selection.json").read_text())
    floor = selection["official_accuracy_floor"]
    picked, basis = selection["selected"], selection["preprocessing"]
    baseline = runs["P00_raw"]
    result = runs["R42_selected"]
    count = sum(len(run["official"]) for run in runs.values())
    lines = [
        f"# Extra Trees {mode} 실험일지", "",
        f"공식 Validation 자격선 {100*floor:.4f}% · {len(runs)}개 실행 조건 · {count}회 적합.",
        f"전처리 후속 기준 **{basis}**, 선정 조건 **{picked}**.",
        "세 자리 실험 ID는 실험 로그·폴더·그림에서 공통으로 사용한다.", "",
        "| ID | 변경 가설 | 비교군 | 공식 Acc. | 손글씨 Acc. | 손글씨 F1 | 판정 |",
        "|---|---|---|---:|---:|---:|---|",
    ]
    prep_names = list(PREPROCESSING)
    for name in (*PREPROCESSING, *TUNING):
        run = runs[name]
        idx = prep_names.index(name) if name in PREPROCESSING else None
        ref = basis if idx is None else prep_names[idx-1] if idx else "W02"
        off, hand, f1 = official_average(run), run["hand"]["accuracy"], run["hand"]["macro_f1"]
        if name == "P00_raw":
            decision = "자격선 산정 기준"
        elif name == picked:
            decision = "최종 채택"
        elif name == basis:
            decision = "전처리 후속 기준"
        elif off < floor:
            decision = "공식 자격선 미달"
        else:
            decision = f"채택군 대비 Macro F1 {f1-result['hand']['macro_f1']:+.4f}"
        lines.append(f"| {name} | {HYPOTHESES[name]} | {ref} | {100*off:.4f}% | "
                     f"{100*hand:.4f}% | {f1:.5f} | {decision} |")
    lines += ["", "## 해석", "",
              "전처리·튜닝 후 비교는 동일한 공식 Validation과 작성자 평가 묶음에서 수행했다.",
              f"시드 42에서 기준 대비 공식 Accuracy "
              f"{100*(official_average(result)-official_average(baseline)):+.4f}%p, "
              f"손글씨 Accuracy {100*(result['hand']['accuracy']-baseline['hand']['accuracy']):+.4f}%p.",
              "동일 시드로 다시 적합한 모델의 공식 및 작성자별 예측 배열이 선정 탐색 모델과 일치했다.",
              "시드 17·42·73 반복과 20~100% 학습곡선을 완료했다.", ""]
    if mode == "Combined":
        control = runs["CTRL_normal"]
        change = result["hand"]["accuracy"]-control["hand"]["accuracy"]
        lines += [
            "## 동일 설정의 자료 추가 대조", "",
            f"공식만 학습 {100*control['hand']['accuracy']:.4f}% → "
            f"손글씨 추가 {100*result['hand']['accuracy']:.4f}% ({100*change:+.4f}%p).",
            "동일 시드·전처리·설정에서 보류 작성자별 동일 이미지를 예측했다. "
            "작성자별 개선과 악화 건수는 matched_writers.csv에 있다.", "",
        ]
    lines += ["## 남은 검증", "",
              "별도 신규 작성자 평가, Original Test와 교수님 Test는 미실시다. "
              "학습 조건 탐색은 일변수 범위이며 조합 전수 탐색이나 신규 480장 보강 비교는 수행하지 않았다.", ""]
    output.write_text("\n".join(lines), encoding="utf-8")


if __name__ == "__main__":
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--mode", choices=("Normal", "Combined"), required=True)
    parser.add_argument("--source", type=Path, required=True)
    parser.add_argument("--output", type=Path, required=True)
    args = parser.parse_args()
    write_log(args.source, args.output, args.mode)


## 실험 실행 및 결과 확인

`RUN_PLAN`에 실제 코드의 CLI 인자와 실행 순서를 넣고, `ALLOW_EXECUTION=True`를 설정한 경우에만 실행된다. 이전 결과를 다시 계산한 것과 새 학습 결과를 구별한다.

In [ ]:
if RUN_PLAN:
    for source_path, arguments in RUN_PLAN:
        execute_original(source_path, arguments)
else:
    print("원본 코드 등록 완료. 학습/평가는 실행하지 않았습니다.")